# 04 — Simple anomaly models

This notebook compares exactly four transparent models:

1. a robust statistical baseline;
2. Isolation Forest;
3. PCA reconstruction error (SPE);
4. PCA Hotelling T-squared.

Each model uses the same causal features. Three calibration thresholds and one
frozen persistence value produce twelve operating points. VUS-PR separately
compares ranking quality without choosing one operating threshold.

Unlike the earlier demo, this notebook does not resample every sector to hourly
data, read ticket labels while fitting, or treat clustering as root-cause truth.

## 1. Setup

In [ ]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DATA_ROOT = Path(
    os.getenv("ANOMALY_DATA_ROOT")
    or os.getenv("ANOMALY_DRIVE_ROOT")
    or (
        "/content/drive/MyDrive/anomaly_detection"
        if IN_COLAB else Path.home() / "anomaly_detection_data"
    )
).expanduser()
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DATA_ROOT / "research" / "milestone1" if IN_COLAB
    else Path.cwd() if (Path.cwd() / "milestone1_core.py").is_file()
    else Path.cwd() / "notebooks" / "drive_research",
)).expanduser()
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

SECTOR = os.getenv("ANOMALY_SECTOR", "telecom")
CANONICAL_RUN_IDS = {
    "telecom": "telecom_core_v0_10_1_run1",
    "petrobras_3w": "petrobras_3w_core_v0_10_1_run1",
}
if SECTOR not in CANONICAL_RUN_IDS:
    raise ValueError(f"Choose one of {list(CANONICAL_RUN_IDS)}")

import tempfile

import duckdb
import joblib
import matplotlib.pyplot as plt

from milestone1_core import CORE_VERSION, new_output_directory, read_json, write_json
from evaluation_core import evaluate_alerts, evaluate_vus_pr
from simple_model_core import (
    MODEL_CORE_VERSION, MODEL_IDS, alerts_from_score_file,
    calibration_thresholds, candidate_key, fit_isolation_seed_models,
    fit_model_bundle, materialize_candidate_alert_grid, materialize_features,
    materialize_wide_partition, partition_exposure, score_feature_file,
    score_isolation_seed_file, score_quantile_thresholds,
)

EDA_VERSION = "1.3.0"
EVALUATION_VERSION = "1.3.0"
MODEL_VERSION = "1.3.0"
CANONICAL_RUN_ID = os.getenv("CANONICAL_RUN_ID", CANONICAL_RUN_IDS[SECTOR])
EDA_RUN_ID = os.getenv("EDA_RUN_ID", f"{SECTOR}_eda_v1_3_run1")
EVALUATION_RUN_ID = os.getenv("EVALUATION_RUN_ID", f"{SECTOR}_evaluation_v1_3_run1")
MODEL_RUN_ID = os.getenv("MODEL_RUN_ID", f"{SECTOR}_models_v1_3_run1")
RUN_ROOT = DATA_ROOT / "outputs" / "canonical" / f"v{CORE_VERSION}" / SECTOR / CANONICAL_RUN_ID
CORE_ROOT, SPLIT_ROOT = RUN_ROOT / "SPEC-CORE", RUN_ROOT / "SPLITS"
EDA_ROOT = DATA_ROOT / "outputs" / "eda" / f"v{EDA_VERSION}" / SECTOR / EDA_RUN_ID
EVALUATION_ROOT = DATA_ROOT / "outputs" / "evaluation" / f"v{EVALUATION_VERSION}" / SECTOR / EVALUATION_RUN_ID
MODEL_ROOT = DATA_ROOT / "outputs" / "models" / f"v{MODEL_VERSION}" / SECTOR / MODEL_RUN_ID
MAX_TRAINING_ROWS = int(os.getenv("MODEL_MAX_TRAINING_ROWS", "100000"))
MINIMUM_THRESHOLD_BLOCK_ROWS = int(os.getenv("MINIMUM_THRESHOLD_BLOCK_ROWS", "1000"))
N_VUS_THRESHOLDS = int(os.getenv("N_VUS_THRESHOLDS", "50"))
N_VUS_TOLERANCES = int(os.getenv("N_VUS_TOLERANCES", "5"))
ISOLATION_SEEDS = [11, 23, 42, 67, 89]
RANDOM_SEED = 42

display(pd.Series({
    "sector": SECTOR, "canonical_input": str(CORE_ROOT),
    "evaluation_contract": str(EVALUATION_ROOT),
    "model_output": str(MODEL_ROOT),
    "candidate_configurations": "defined by the frozen evaluation policy",
}, name="value").to_frame())

## 2. Load the frozen decisions and development truth

In [ ]:
manifest = read_json(CORE_ROOT / "manifest.json")
catalogue = pd.read_parquet(CORE_ROOT / "metric_catalogue.parquet")
decisions = read_json(EDA_ROOT / "eda_decisions.json")
eda_summary = pd.read_csv(EDA_ROOT / "eda_summary.csv")
policy = read_json(EVALUATION_ROOT / "evaluation_policy.json")
entity_groups_path = SPLIT_ROOT / "entity_groups.parquet"
entity_groups = (
    pd.read_parquet(entity_groups_path)
    if entity_groups_path.is_file() else pd.DataFrame()
)
development_truth = EVALUATION_ROOT / "development"
fault_events = pd.read_parquet(development_truth / "fault_events.parquet")
fault_intervals = pd.read_parquet(development_truth / "fault_entity_intervals.parquet")

if manifest["sector"] != SECTOR:
    raise ValueError("Canonical sector does not match the requested sector")
if decisions["eda_version"] != EDA_VERSION:
    raise ValueError("EDA decisions are not from the declared EDA version")
if policy["evaluation_version"] != EVALUATION_VERSION:
    raise ValueError("Evaluation policy is not from the declared version")
if decisions["canonical_fingerprint"] != manifest["fingerprint"]:
    raise ValueError("EDA decisions belong to a different canonical run")
if policy["canonical_fingerprint"] != manifest["fingerprint"]:
    raise ValueError("Evaluation policy belongs to a different canonical run")
canonical_build_ts = pd.Timestamp(
    (CORE_ROOT / "manifest.json").stat().st_mtime, unit="s", tz="UTC"
)
display(pd.Series({
    "canonical_run_id": CANONICAL_RUN_ID,
    "canonical_build_ts": canonical_build_ts,
}, name="value").to_frame())

cadences = decisions["cadence_values_seconds"]
if len(cadences) != 1:
    raise ValueError(
        "The current multivariate baseline requires aligned metric cadence. "
        "The canonical contract supports mixed cadence, but this model does not "
        "silently resample it."
    )
calibration_clipped_rate = np.average(
    eda_summary["clipped_rate"].fillna(0),
    weights=eda_summary["rows"],
)

display(pd.Series({
    "calibration_only_fit": True,
    "development_faults": len(fault_events),
    "holdout_read": False,
    "rolling_window_observations": decisions["rolling_window_observations"],
    "persistence_observations": policy["persistence_observations"],
    "candidate_configurations": (
        len(MODEL_IDS) * len(policy["threshold_quantiles"])
    ),
}, name="value").to_frame())

## 3. Build causal features and fit on calibration only

In [ ]:
work = tempfile.TemporaryDirectory(prefix=f"{SECTOR}-simple-model-")
WORK_ROOT = Path(work.name)
lookback_seconds = max(decisions["rolling_windows_seconds"].values())

paths = {}
for partition in ("calibration", "development"):
    started = time.perf_counter()
    wide = WORK_ROOT / f"{partition}_wide.parquet"
    bounds = materialize_wide_partition(
        CORE_ROOT, SPLIT_ROOT, partition, catalogue, wide,
        lookback_seconds=lookback_seconds,
        memory_limit=os.getenv("ANOMALY_DUCKDB_MEMORY_LIMIT", "3GB"),
        threads=int(os.getenv("ANOMALY_DUCKDB_THREADS", "2")),
    )
    features = WORK_ROOT / f"{partition}_features.parquet"
    materialize_features(
        wide, catalogue, decisions, features,
        score_start=bounds["score_start"], score_end=bounds["score_end"],
    )
    paths[partition] = {"wide": wide, "features": features}
    print(f"Prepared {partition} features in {(time.perf_counter() - started) / 60:.1f} min")

started = time.perf_counter()
bundle = fit_model_bundle(
    paths["calibration"]["features"], decisions,
    maximum_training_rows=MAX_TRAINING_ROWS,
    random_seed=RANDOM_SEED,
)
for partition in paths:
    score_path = WORK_ROOT / f"{partition}_scores.parquet"
    score_feature_file(bundle, paths[partition]["features"], score_path)
    paths[partition]["scores"] = score_path
print(f"Fitted and scored four models in {(time.perf_counter() - started) / 60:.1f} min")

print("Calibration training rows:", bundle["training_rows"])
print("Usable features:", len(bundle["feature_columns"]))
print("PCA components:", bundle["pca"].n_components_)
display(pd.DataFrame({"feature": bundle["feature_columns"]}))

## 4. Compare the development configurations

In [ ]:
persistence = int(policy["persistence_observations"])
threshold_block = "episode_id" if SECTOR == "petrobras_3w" else "entity_id"
operating_thresholds = calibration_thresholds(
    paths["calibration"]["scores"],
    policy["threshold_quantiles"],
    block_column=threshold_block,
    minimum_block_rows=MINIMUM_THRESHOLD_BLOCK_ROWS,
)
operating_thresholds["purpose"] = "operating"
operating_thresholds["candidate_key"] = [
    f"operating|{candidate_key(row.model_id, row.threshold_quantile, persistence)}"
    for row in operating_thresholds.itertuples(index=False)
]

tail_probabilities = np.geomspace(0.10, 0.0001, N_VUS_THRESHOLDS)
vus_quantiles = 1 - tail_probabilities
vus_thresholds = score_quantile_thresholds(
    paths["development"]["scores"], vus_quantiles
)
vus_thresholds["purpose"] = "vus"
vus_thresholds["candidate_key"] = [
    f"vus|{row.model_id}|{number:03d}"
    for number, row in enumerate(vus_thresholds.itertuples(index=False), start=1)
]
all_thresholds = pd.concat(
    [operating_thresholds, vus_thresholds], ignore_index=True, sort=False
)
exposure = partition_exposure(
    paths["development"]["scores"],
    policy["primary_exposure_unit"],
    decisions["base_cadence_seconds"],
)
false_metric = f"false_alerts_per_{policy['primary_exposure_unit']}"

started = time.perf_counter()
candidate_manifest = materialize_candidate_alert_grid(
    paths["development"]["scores"],
    all_thresholds, [persistence],
    WORK_ROOT / "candidate_alerts",
    recovery_consecutive=policy["recovery_observations"],
)
operating_manifest = candidate_manifest.loc[
    candidate_manifest["purpose"].eq("operating")
]
threshold_provenance = operating_thresholds.set_index("candidate_key")
vus_manifest = candidate_manifest.loc[
    candidate_manifest["purpose"].eq("vus")
]
rows, fault_type_rows = [], []
current_model = None
for candidate in operating_manifest.itertuples(index=False):
    if candidate.model_id != current_model:
        current_model = candidate.model_id
        print(f"Evaluating {current_model} configurations")
    threshold_row = threshold_provenance.loc[candidate.candidate_key]
    alerts = pd.read_parquet(candidate.alert_path)
    result = evaluate_alerts(
        alerts, fault_events, fault_intervals,
        exposure_value=exposure,
        exposure_unit=policy["primary_exposure_unit"],
        decision_horizon_seconds=policy["decision_horizon_seconds"],
        grouping_window_seconds=policy["grouping_window_seconds"],
        entity_groups=entity_groups,
    )
    metric_table = result["metrics"].set_index("metric")
    metrics = metric_table["value"].to_dict()
    rows.append({
        "candidate_key": candidate.candidate_key,
        "alert_path": str(candidate.alert_path),
        "model_id": candidate.model_id,
        "threshold_quantile": candidate.threshold_quantile,
        "threshold": candidate.threshold,
        "threshold_block": threshold_row.threshold_block,
        "blocks_total": threshold_row.blocks_total,
        "blocks_used": threshold_row.blocks_used,
        "blocks_excluded": threshold_row.blocks_excluded,
        "persistence_observations": candidate.persistence_observations,
        "recovery_observations": int(policy["recovery_observations"]),
        "alert_count": candidate.alert_count,
        "event_recall": metrics["event_recall"],
        "event_recall_ci_low": metric_table.loc["event_recall", "ci_low"],
        "event_recall_ci_high": metric_table.loc["event_recall", "ci_high"],
        "preimpact_event_recall": metrics["preimpact_event_recall"],
        "alert_precision": metrics["alert_precision"],
        "false_alert_rate": metrics[false_metric],
        "false_alert_ci_low": metric_table.loc[false_metric, "ci_low"],
        "false_alert_ci_high": metric_table.loc[false_metric, "ci_high"],
        "false_alert_clusters": metrics["false_alert_cluster_count"],
        "duplicate_alerts": metrics["duplicate_alerts"],
        "median_delay_seconds": metrics["median_detection_delay_seconds"],
        "calibration_clipped_rate": calibration_clipped_rate,
    })
    by_type = result["fault_type_results"].copy()
    by_type.insert(0, "candidate_key", candidate.candidate_key)
    by_type.insert(1, "model_id", candidate.model_id)
    by_type.insert(2, "threshold_quantile", candidate.threshold_quantile)
    by_type.insert(3, "persistence_observations", persistence)
    fault_type_rows.append(by_type)

vus_points, vus_summary = evaluate_vus_pr(
    vus_manifest, fault_events, fault_intervals,
    exposure_value=exposure,
    exposure_unit=policy["primary_exposure_unit"],
    decision_horizon_seconds=policy["decision_horizon_seconds"],
    grouping_window_seconds=policy["grouping_window_seconds"],
    entity_groups=entity_groups,
    n_tolerances=N_VUS_TOLERANCES,
)
print(f"Evaluated the complete grid in {(time.perf_counter() - started) / 60:.1f} min")

comparison = pd.DataFrame(rows).merge(
    vus_summary, on="model_id", how="left", validate="many_to_one"
)
fault_type_comparison = pd.concat(fault_type_rows, ignore_index=True)
comparison["budget_point_compliant"] = comparison["false_alert_rate"].le(
    policy["false_alert_budget"]
)
comparison["budget_95_compliant"] = comparison["false_alert_ci_high"].le(
    policy["false_alert_budget"]
)
comparison["nonzero_alerts"] = comparison["alert_count"].gt(0)
expected_candidates = (
    len(MODEL_IDS) * len(policy["threshold_quantiles"])
)
assert len(comparison) == expected_candidates
assert len(comparison) == 12
display(vus_summary.round(4))
display(comparison.round(4))

## 5. Apply the predeclared selection rule

In [ ]:
def budget_pool(table):
    eligible = table.loc[table["nonzero_alerts"]].copy()
    confidence = eligible.loc[eligible["budget_95_compliant"]]
    point = eligible.loc[eligible["budget_point_compliant"]]
    if not confidence.empty:
        return confidence, "budget_qualified_at_95_percent"
    if not point.empty:
        return point, "budget_qualified_by_point_estimate_only"
    return eligible, "research_candidate_not_budget_qualified"

if policy["primary_selection_metric"] == "preimpact_event_recall":
    selection_columns = [
        "preimpact_event_recall", "event_recall", "alert_precision",
        "median_delay_seconds", "alert_count",
    ]
    selection_order = [False, False, False, True, True]
else:
    selection_columns = [
        "event_recall", "alert_precision",
        "median_delay_seconds", "alert_count",
    ]
    selection_order = [False, False, True, True]

operating_pool, operating_status = budget_pool(comparison)
if operating_pool.empty:
    raise ValueError("Every candidate produced zero alerts")
operating_winner = operating_pool.sort_values(
    selection_columns, ascending=selection_order,
    na_position="last",
).iloc[0]
vus_winner = vus_summary.iloc[0]["model_id"]
usable_models = set(comparison.loc[comparison["nonzero_alerts"], "model_id"])
vus_operable = vus_summary.loc[vus_summary["model_id"].isin(usable_models)]
if vus_operable.empty:
    raise ValueError("No VUS-PR model has a usable operating point")
selected_model_by_vus = vus_operable.iloc[0]["model_id"]
model_pool, selection_status = budget_pool(
    comparison.loc[comparison["model_id"].eq(selected_model_by_vus)]
)
if model_pool.empty:
    raise ValueError("The VUS-PR winner has no usable operating point")
selected = model_pool.sort_values(
    selection_columns, ascending=selection_order,
    na_position="last",
).iloc[0]
selected_key = selected["candidate_key"]
selected_alert_path = comparison.loc[
    comparison["candidate_key"].eq(selected_key), "alert_path"
].iloc[0]
development_alerts = pd.read_parquet(selected_alert_path).sort_values(
    ["alert_start", "entity_id", "episode_id"]
).reset_index(drop=True)
development_alerts["alert_id"] = [
    f"A-{number:09d}"
    for number in range(1, len(development_alerts) + 1)
]
selected_result = evaluate_alerts(
    development_alerts, fault_events, fault_intervals,
    exposure_value=exposure,
    exposure_unit=policy["primary_exposure_unit"],
    decision_horizon_seconds=policy["decision_horizon_seconds"],
    grouping_window_seconds=policy["grouping_window_seconds"],
    entity_groups=entity_groups,
)
print("VUS-PR winner:", vus_winner)
print("Highest-VUS model with an operating point:", selected_model_by_vus)
if vus_winner != selected_model_by_vus:
    print("FINDING — the VUS-PR winner has no usable frozen operating point")
print("Operating-point winner:", operating_winner["model_id"])
if vus_winner != operating_winner["model_id"]:
    print("FINDING — threshold-free and operating-point rankings disagree")

configuration = {
    "model_version": MODEL_VERSION,
    "model_core_version": MODEL_CORE_VERSION,
    "scaled_feature_cap": float(bundle["scaled_feature_cap"]),
    "pca_variance_target": float(bundle["pca_variance_target"]),
    "pca_spe_tail_ratio": float(bundle["pca_spe_tail_ratio"]),
    "sector": SECTOR,
    "canonical_fingerprint": manifest["fingerprint"],
    "model_id": selected["model_id"],
    "threshold_quantile": float(selected["threshold_quantile"]),
    "threshold": float(selected["threshold"]),
    "threshold_block": selected["threshold_block"],
    "threshold_blocks_total": int(selected["blocks_total"]),
    "threshold_blocks_used": int(selected["blocks_used"]),
    "threshold_blocks_excluded": int(selected["blocks_excluded"]),
    "threshold_guarantee": (
        "Controls the point-level exceedance rate marginally across "
        "calibration blocks. It does not guarantee an alert rate under "
        "temporal dependence; the empirical alert rate and clustered "
        "interval are reported separately."
    ),
    "persistence_observations": int(selected["persistence_observations"]),
    "recovery_observations": int(selected["recovery_observations"]),
    "selection_status": selection_status,
    "primary_exposure_unit": policy["primary_exposure_unit"],
    "primary_selection_metric": policy["primary_selection_metric"],
    "false_alert_budget": policy["false_alert_budget"],
    "false_alert_rate": float(selected["false_alert_rate"]),
    "false_alert_ci_high": float(selected["false_alert_ci_high"]),
    "decision_horizon_seconds": policy["decision_horizon_seconds"],
    "development_exposure_value": float(exposure),
    "vus_pr": float(selected["vus_pr"]),
    "frozen_window_map": decisions["rolling_windows_seconds"],
    "feature_settings": decisions,
    "candidate_count": len(comparison),
    "holdout_used": False,
    "leading_feature_note": {
        "statistical": "largest absolute robust feature deviation",
        "isolation_forest": "largest absolute robust feature deviation; explanation proxy",
        "pca": "largest PCA reconstruction residual",
        "pca_t2": "largest absolute Hotelling T-squared contribution",
    }[selected["model_id"]],
}
bundle["selected_model_id"] = selected["model_id"]

display(pd.Series(configuration, name="value").to_frame())

figure, axes = plt.subplots(1, 2, figsize=(12, 4))
for model_id, group in comparison.groupby("model_id"):
    axes[0].scatter(group["false_alert_rate"], group["event_recall"], label=model_id)
    axes[1].scatter(group["alert_precision"], group["event_recall"], label=model_id)
axes[0].axvline(policy["false_alert_budget"], color="red", linestyle="--")
axes[0].set(xlabel=false_metric, ylabel="event recall", title="Recall versus false-alert workload")
axes[1].set(xlabel="alert precision", ylabel="event recall", title="Recall versus precision")
axes[0].legend(); axes[1].legend()
figure.tight_layout(); plt.show()

figure, axis = plt.subplots(figsize=(6, 4))
maximum_tolerance = vus_points["tolerance_seconds"].max()
for model_id, curve in vus_points.loc[
    vus_points["tolerance_seconds"].eq(maximum_tolerance)
].groupby("model_id"):
    curve = curve.sort_values("event_recall")
    axis.plot(curve["event_recall"], curve["alert_precision"], label=model_id)
axis.set(xlabel="event recall", ylabel="alert precision", title="Development PR curves")
axis.legend()
figure.tight_layout(); plt.show()

## 6. Stability, warm-up and coverage diagnostics

In [ ]:
seed_models = fit_isolation_seed_models(
    bundle, paths["calibration"]["features"], ISOLATION_SEEDS,
    maximum_training_rows=MAX_TRAINING_ROWS, sample_seed=RANDOM_SEED,
)
seed_calibration_scores = WORK_ROOT / "seed_calibration_scores.parquet"
seed_development_scores = WORK_ROOT / "seed_development_scores.parquet"
score_isolation_seed_file(
    bundle, seed_models, paths["calibration"]["features"],
    seed_calibration_scores,
)
score_isolation_seed_file(
    bundle, seed_models, paths["development"]["features"],
    seed_development_scores,
)
seed_thresholds = calibration_thresholds(
    seed_calibration_scores, [float(selected["threshold_quantile"])],
    block_column=threshold_block,
    minimum_block_rows=MINIMUM_THRESHOLD_BLOCK_ROWS,
    model_ids=tuple(seed_models),
)
seed_rows = []
for threshold in seed_thresholds.itertuples(index=False):
    seed_alerts = alerts_from_score_file(
        seed_development_scores, threshold.model_id, threshold.threshold,
        min_consecutive=persistence,
        recovery_consecutive=policy["recovery_observations"],
    )
    seed_result = evaluate_alerts(
        seed_alerts, fault_events, fault_intervals,
        exposure_value=exposure,
        exposure_unit=policy["primary_exposure_unit"],
        decision_horizon_seconds=policy["decision_horizon_seconds"],
        grouping_window_seconds=policy["grouping_window_seconds"],
        entity_groups=entity_groups,
    )
    value = seed_result["metrics"].set_index("metric").loc[
        policy["primary_selection_metric"], "value"
    ]
    seed_rows.append({
        "seed": int(threshold.model_id.rsplit("_", 1)[-1]),
        "primary_metric": policy["primary_selection_metric"],
        "primary_metric_value": value,
    })
seed_stability = pd.DataFrame(seed_rows)
other_models = comparison.loc[
    comparison["model_id"].ne("isolation_forest")
].sort_values(policy["primary_selection_metric"], ascending=False)
if other_models.empty:
    seed_stability["operating_rank_winner"] = "isolation_forest"
else:
    comparator = other_models.iloc[0]
    seed_stability["operating_rank_winner"] = np.where(
        seed_stability["primary_metric_value"].ge(
            comparator[policy["primary_selection_metric"]]
        ),
        "isolation_forest", comparator["model_id"],
    )
seed_spread = (
    seed_stability["primary_metric_value"].max()
    - seed_stability["primary_metric_value"].min()
)
seed_ranking_stable = seed_stability["operating_rank_winner"].nunique() == 1
display(seed_stability)
print("Isolation Forest primary-metric spread:", round(seed_spread, 4))
print("Operating ranking stable across seeds:", seed_ranking_stable)

score_source = str(paths["development"]["scores"]).replace("'", "''")
connection = duckdb.connect()
episode_starts = connection.execute(f"""
    SELECT entity_id, episode_id, min(event_ts) AS episode_start
    FROM read_parquet('{score_source}')
    GROUP BY entity_id, episode_id
""").df()
detected = selected_result["fault_results"].loc[
    selected_result["fault_results"]["detected"]
].merge(
    development_alerts[["alert_id", "entity_id", "episode_id"]],
    on="alert_id", how="left", validate="one_to_one",
).merge(
    episode_starts, on=["entity_id", "episode_id"], how="left",
)
detected["time_since_episode_start_seconds"] = (
    pd.to_datetime(detected["alert_start"], utc=True)
    - pd.to_datetime(detected["episode_start"], utc=True)
).dt.total_seconds()
warmup_bias = detected[[
    "fault_id", "fault_type", "entity_id", "episode_id",
    "detection_delay_seconds", "time_since_episode_start_seconds",
]].copy()
warmup_detections = warmup_bias["time_since_episode_start_seconds"].lt(
    decisions["minimum_history_seconds"]
).sum()
warmup_limitation = bool(len(warmup_bias) and warmup_detections == 0)
figure, axis = plt.subplots(figsize=(6, 4))
axis.scatter(
    warmup_bias["time_since_episode_start_seconds"],
    warmup_bias["detection_delay_seconds"], alpha=0.7,
)
axis.axvline(decisions["minimum_history_seconds"], color="red", linestyle="--")
axis.set(
    xlabel="seconds since episode start",
    ylabel="detection delay (seconds)", title="Warm-up diagnostic",
)
figure.tight_layout(); plt.show()
print("Warm-up blind-spot limitation:", warmup_limitation)

wide_source = str(paths["development"]["wide"]).replace("'", "''")
quoted_metrics = [
    '"' + metric.replace('"', '""') + '"'
    for metric in catalogue["metric_id"].astype(str)
]
presence_terms = [
    f"CASE WHEN count({metric}) FILTER (WHERE {metric} IS NOT NULL) > 0 "
    "THEN 1 ELSE 0 END"
    for metric in quoted_metrics
]
coverage = connection.execute(f"""
    SELECT entity_id, {' + '.join(presence_terms)} AS observed_metric_count
    FROM read_parquet('{wide_source}') GROUP BY entity_id
""").df()
episode_exposure = connection.execute(f"""
    SELECT entity_id, episode_id, min(event_ts) AS first_ts,
           max(event_ts) AS last_ts
    FROM read_parquet('{score_source}') GROUP BY entity_id, episode_id
""").df()
episode_seconds = (
    pd.to_datetime(episode_exposure["last_ts"], utc=True)
    - pd.to_datetime(episode_exposure["first_ts"], utc=True)
).dt.total_seconds() + decisions["base_cadence_seconds"]
if policy["primary_exposure_unit"] == "episode":
    entity_exposure = episode_exposure.groupby("entity_id").size()
elif policy["primary_exposure_unit"] == "entity_day":
    entity_exposure = episode_seconds.groupby(episode_exposure["entity_id"]).sum() / 86400
else:
    entity_exposure = episode_seconds.groupby(episode_exposure["entity_id"]).sum() / 3600
alert_counts = development_alerts.groupby("entity_id").size()
coverage_heterogeneity = coverage.copy()
coverage_heterogeneity["exposure"] = coverage_heterogeneity["entity_id"].map(entity_exposure)
coverage_heterogeneity["alert_count"] = coverage_heterogeneity["entity_id"].map(alert_counts).fillna(0)
coverage_heterogeneity["alert_rate"] = (
    coverage_heterogeneity["alert_count"] / coverage_heterogeneity["exposure"]
)
coverage_alert_correlation = coverage_heterogeneity[[
    "observed_metric_count", "alert_rate"
]].corr(method="spearman").iloc[0, 1]
figure, axis = plt.subplots(figsize=(6, 4))
axis.scatter(
    coverage_heterogeneity["observed_metric_count"],
    coverage_heterogeneity["alert_rate"], alpha=0.7,
)
axis.set(
    xlabel="observed metrics per entity",
    ylabel=f"alerts per {policy['primary_exposure_unit']}",
    title="Coverage heterogeneity",
)
figure.tight_layout(); plt.show()
print("Coverage-alert Spearman correlation:", round(coverage_alert_correlation, 4))
connection.close()

## 7. Save the selected model and its development evidence

In [ ]:
with new_output_directory(MODEL_ROOT) as output:
    configuration["seed_primary_metric_spread"] = float(seed_spread)
    configuration["seed_operating_ranking_stable"] = bool(seed_ranking_stable)
    configuration["warmup_blind_spot_limitation"] = warmup_limitation
    configuration["coverage_alert_spearman"] = (
        float(coverage_alert_correlation)
        if pd.notna(coverage_alert_correlation) else None
    )
    comparison.drop(columns="alert_path").to_csv(
        output / "model_comparison.csv", index=False
    )
    fault_type_comparison.to_csv(
        output / "model_comparison_by_fault_type.csv", index=False
    )
    vus_points.to_csv(output / "vus_pr_points.csv", index=False)
    vus_summary.to_csv(output / "vus_pr_summary.csv", index=False)
    seed_stability.to_csv(output / "seed_stability.csv", index=False)
    warmup_bias.to_csv(output / "warmup_bias.csv", index=False)
    coverage_heterogeneity.to_csv(
        output / "coverage_heterogeneity.csv", index=False
    )
    development_alerts.to_parquet(
        output / "development_alerts.parquet", index=False
    )
    joblib.dump(bundle, output / "selected_model.joblib")
    write_json(output / "selected_configuration.json", configuration)
    write_json(output / "holdout_prediction_template.json", {
        "instructions": (
            "Copy to holdout_prediction.json and pre-register an expected "
            "value, confirmation_min and confirmation_max for every "
            "primary metric before RUN_HOLDOUT=1."
        ),
        "predictions": {
            policy["primary_selection_metric"]: {
                "expected": None, "confirmation_min": None,
                "confirmation_max": None,
            },
            false_metric: {
                "expected": None, "confirmation_min": None,
                "confirmation_max": None,
            },
        },
    })

print("Saved:", MODEL_ROOT)
print("Selected model:", configuration["model_id"])
print("Selection status:", selection_status)
print("Next: 05_INCIDENT_RANKING_AND_HOLDOUT.ipynb")
work.cleanup()